# C8 — Reports (parent PDF + dev report + delivery hooks)

Implements PROJECT_BRIEF C8. Loaded as `geeta.reports` by notebooks and the webapp server.

| Piece | What it does |
|-------|--------------|
| `derive_metrics` | Turns a `TutorSession` (or its `close()` dict) into all report metrics: time, confidence, engagement, verifier pass rate, hint depth, retries, leak triggers, OCR corrections |
| `ParentReport` | Kid-friendly PDF (reportlab + matplotlib charts): what was learned, time taken, confidence, engagement |
| `DevReport` | JSON dev metrics: tokens/s, TTFT, verifier pass rate, hint depth, retries, leak-guard triggers |
| `send_email_dryrun` / `send_whatsapp_dryrun` | Build the exact delivery payloads and write them to disk. **Never sends** (no network, no credentials) — swap in SES/SMTP + WhatsApp Business API at pilot time |

Privacy: reports contain the child's first name only (delivered to their parent). Nothing else is persisted.


In [ ]:
# --- GEETA notebook-module bootstrap (idempotent; makes tools/*.ipynb importable as geeta.*) ---
import json as _json
import pathlib as _pathlib

_ROOT = _pathlib.Path.cwd()
if not (_ROOT / "tools" / "_nbmodule.ipynb").exists():      # run from project root or tools/
    _ROOT = _ROOT.parent
_nb = _json.loads((_ROOT / "tools" / "_nbmodule.ipynb").read_text())
_src = "".join("".join(c["source"]) for c in _nb["cells"] if c["cell_type"] == "code")
exec(compile(_src, str(_ROOT / "tools" / "_nbmodule.ipynb"), "exec"), globals())
bootstrap(_ROOT)  # noqa: E501


In [ ]:
# ---------------------------------------------------------------------------
# Metric derivation (single source of truth for parent + dev reports)
# ---------------------------------------------------------------------------

import json
from pathlib import Path


def _clamp(x: float, lo: float = 0.0, hi: float = 1.0) -> float:
    return max(lo, min(hi, x))


def derive_metrics(session, cfg: dict) -> dict:
    """Derive all report metrics from a TutorSession or its close() summary.

    Confidence and engagement are DOCUMENTED HEURISTICS (brief C8:
    "confidence (verifier/hint-depth derived), engagement (session events)").
    They are proxies for the parent, not model-quality claims.
    """
    if hasattr(session, "close"):
        s = session.close()
    else:
        s = dict(session)

    events = s.get("events", [])
    dur = float(s.get("duration_s") or 0.0)
    hints = [e for e in events if e.get("kind") == "hint"]
    reeng = [e for e in events if e.get("kind") == "reengage"]
    guards = [e for e in events if e.get("kind") == "guard"]
    verifs = [e for e in events if e.get("kind") == "verifier"]
    ocr_conf = [e for e in events if e.get("kind") == "ocr_confirm"]
    inputs = [e for e in events if e.get("kind") == "input"]

    n_turns = max(len(inputs), 1)
    minutes = max(dur / 60.0, 1e-3)
    turns_per_min = len(inputs) / minutes

    verifier_ok = [e for e in verifs if e.get("ok")]
    verifier_pass_rate = (len(verifier_ok) / len(verifs)) if verifs else None

    max_depth = int(s.get("max_hint_depth") or 0)
    leak_triggers = sum(1 for e in guards if e.get("trigger"))
    ocr_corrections = sum(1 for e in ocr_conf if not e.get("accepted"))

    # confidence: start high; each hint depth the kid needed costs a little;
    # re-engagements cost more; a clean verifier pass adds back.
    conf = 0.95 - 0.10 * max_depth - 0.10 * len(reeng)
    if verifier_pass_rate is not None:
        conf += 0.10 * (verifier_pass_rate - 0.8)
    confidence = round(_clamp(conf, 0.2, 0.95), 3)

    # engagement: turn rate + penalty for drift (re-engagements per turn)
    reengage_rate = len(reeng) / n_turns
    engagement = round(_clamp(0.45 + 0.12 * min(turns_per_min, 6.0) - 0.35 * reengage_rate, 0.05, 1.0), 3)

    return {
        "kid": s.get("kid"),
        "problem_id": s.get("problem_id"),
        "skill_id": s.get("skill_id"),
        "year": s.get("year"),
        "strand": s.get("strand"),
        "solved": bool(s.get("solved")),
        "solved_how": s.get("solved_how"),
        "duration_s": round(dur, 1),
        "hints_used": len(hints),
        "max_hint_depth": max_depth,
        "reengagements": len(reeng),
        "attempts": s.get("attempts"),
        "turns": len(inputs),
        "turns_per_min": round(turns_per_min, 2),
        "confidence": confidence,
        "engagement": engagement,
        "verifier_pass_rate": (round(verifier_pass_rate, 3) if verifier_pass_rate is not None else None),
        "verifier_checks": len(verifs),
        "leak_triggers": leak_triggers,
        "ocr_corrections_by_kid": ocr_corrections,
        "focus": s.get("focus"),   # None unless the kid opted into the on-device focus helper
        "n_events": len(events),
        "items": s.get("items") or [],   # per worksheet question (skill, solved, hints, mistakes)
        # raw events kept for the PDF charts (not part of the dev JSON)
        "_events": events,
    }



# ---------------------------------------------------------------------------
# ACARA-style achievement grade (session 13, user request): parents already
# read A-E / "working towards - you've done it - above and beyond" rubric
# grids on their child's actual school reports (e.g. IB PYP schools like
# Linden Park Primary). GEETA's numeric confidence/hints/solved metrics are
# mapped into that SAME shape so the report is legible in a format parents
# already know - not because GEETA claims to be a school-endorsed grade.
# Band definitions are our own design (documented in config/eval/rubric.md
# section 4), not copied from any school's rubric text.
_SKILL_MAP = None


def _load_skill_map(cfg: dict) -> dict:
    global _SKILL_MAP
    if _SKILL_MAP is None:
        p = Path(cfg["paths"]["data_bank"]) / "skill_map.json"
        _SKILL_MAP = {s["id"]: s for s in json.loads(p.read_text())["skills"]} if p.exists() else {}
    return _SKILL_MAP


def derive_grade(m: dict, cfg: dict) -> dict:
    """Map session metrics -> {letter A-E, band, attitude, skill_name, acara_code, desc}.

    Letter grade (own design, not a real assessment):
      A - solved, 0 hints, no re-engagement needed         (confidence >= 0.85)
      B - solved, light help (<=1 hint, no reset)          (confidence >= 0.70)
      C - solved with hints and/or one gentle reset         (confidence >= 0.55)
      D - got there via similar-example/story reframe, or multiple resets
      E - fallback used / not solved this session
    Attitude to learning mirrors the same three bands schools commonly use
    (Consistent / Satisfactory / Developing), driven by the engagement proxy.
    """
    conf = m.get("confidence") or 0.0
    solved = bool(m.get("solved"))
    reeng = int(m.get("reengagements") or 0)
    hints = int(m.get("hints_used") or 0)

    if solved and hints == 0 and reeng == 0:
        letter, band = "A", "above_beyond"
    elif solved and hints <= 1 and reeng == 0:
        letter, band = "B", "above_beyond"
    elif solved and conf >= 0.55:
        letter, band = "C", "done_it"
    elif reeng >= 1 or hints >= 2:
        letter, band = "D", "working_towards"
    else:
        letter, band = "E", "working_towards"

    eng = m.get("engagement") or 0.0
    attitude = "Consistent" if eng >= 0.70 else ("Satisfactory" if eng >= 0.40 else "Developing")

    sm = _load_skill_map(cfg).get(m.get("skill_id") or "", {})
    return {
        "letter": letter,
        "band": band,       # working_towards | done_it | above_beyond
        "attitude": attitude,
        "skill_name": sm.get("name", m.get("skill_id") or _items_skill_name(m) or "today's skill"),
        "acara_code": sm.get("code"),
        "desc": sm.get("desc", ""),
    }


def _items_skill_name(m: dict) -> str | None:
    names = list(dict.fromkeys(it["skill"] for it in (m.get("items") or [])))
    return "; ".join(names) if names else None


def _band_descriptor(desc: str, band: str) -> str:
    """Templated 3-tier phrasing for one ACARA skill (own wording, not a
    real school's rubric text - see derive_grade docstring)."""
    if not desc:      # homework-page skills have no ACARA descriptor text
        return {"working_towards": "Working on it, with GEETA's hints and worked examples.",
                "done_it": "Can do it, with a little help along the way.",
                "above_beyond": "Does it confidently and independently."}[band]
    stem = desc[0].lower() + desc[1:]
    if band == "working_towards":
        return f"Tries to {stem}, with GEETA's hints and worked examples."
    if band == "done_it":
        return f"Can {stem}, with a little help along the way."
    return f"Confidently and independently can {stem}."

In [ ]:
# ---------------------------------------------------------------------------
# Parent PDF (reportlab + matplotlib)
# ---------------------------------------------------------------------------

import io
from datetime import datetime, timezone
from xml.sax.saxutils import escape as _x   # reportlab Paragraphs are mini-markup: escape anything user-made

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.units import mm
from reportlab.platypus import Image, Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle


def _t0(events: list[dict]) -> float:
    """Event times are wall-clock seconds; charts show seconds INTO the session."""
    ts = [e.get("t") for e in events if isinstance(e.get("t"), (int, float))]
    return min(ts) if ts else 0.0


def _chart_hint_depth(events: list[dict]) -> bytes | None:
    """Hint depth over the session (step chart)."""
    t0 = _t0(events)
    pts = [(e.get("t", 0.0) - t0, int(e.get("depth", 0))) for e in events if e.get("kind") == "hint"]
    if not pts:
        return None
    ts = [p[0] for p in pts]
    ds = [p[1] for p in pts]
    fig, ax = plt.subplots(figsize=(6.4, 2.2), dpi=150)
    ax.step(ts, ds, where="post", color="#e07a5f", lw=2)
    ax.scatter(ts, ds, color="#e07a5f", s=30, zorder=3)
    ax.set_xlabel("seconds into session")
    ax.set_ylabel("hint depth")
    ax.set_title("How much help was needed", fontsize=10)
    ax.set_ylim(-0.4, max(ds + [1]) + 0.6)
    ax.grid(alpha=0.25)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight")
    plt.close(fig)
    return buf.getvalue()


def _chart_timeline(events: list[dict]) -> bytes | None:
    """Session timeline: kid inputs, hints, re-engagements, solve."""
    marks = {"input": ("^", "#81b298", "you tried"),
             "hint": ("s", "#e07a5f", "hint"),
             "reengage": ("D", "#f2cc8f", "we restarted gently"),
             "solved": ("*", "#3d405b", "solved!")}
    t0 = _t0(events)
    fig, ax = plt.subplots(figsize=(6.4, 2.0), dpi=150)
    for e in events:
        m = marks.get(e.get("kind"))
        if not m:
            continue
        ax.scatter([e.get("t", 0.0) - t0], [0], marker=m[0], color=m[1], s=70, zorder=3)
    ax.set_yticks([])
    ax.set_xlabel("seconds into session")
    ax.set_title("What happened in your session", fontsize=10)
    ax.grid(alpha=0.25, axis="x")
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight")
    plt.close(fig)
    return buf.getvalue()


class ParentReport:
    """Kid-friendly parent PDF: what was learned, time, confidence, engagement."""

    SKILL_PLAIN = {
        "number_place_value": "place value (how numbers are built from 1s, 10s, 100s)",
        "number_addition": "adding numbers",
        "number_subtraction": "taking away / subtracting",
        "number_multiplication": "multiplying (groups of)",
        "number_division": "sharing out / dividing",
        "number_fractions": "fractions (parts of a whole or a set)",
        "algebra_patterns": "spotting patterns and rules",
        "measurement_time": "telling and comparing time",
        "measurement_money": "counting and comparing money",
        "space_shapes": "shapes and their features",
    }

    def __init__(self, session, cfg: dict, history: dict | None = None):
        self.cfg = cfg
        self.m = derive_metrics(session, cfg)
        # portal (2026-10-13): this child's progress across earlier sessions
        self.history = history

    def _skill_line(self) -> str:
        names = list(dict.fromkeys(it["skill"] for it in (self.m.get("items") or [])))
        if names:
            return " and ".join(n[0].lower() + n[1:] for n in names)
        sid = self.m.get("skill_id") or ""
        plain = self.SKILL_PLAIN.get(sid)
        if plain:
            return plain
        if sid:
            return sid.replace("_", " ")
        return "maths skills from their homework"

    def build(self) -> bytes:
        m = self.m
        buf = io.BytesIO()
        doc = SimpleDocTemplate(buf, pagesize=A4,
                                leftMargin=18 * mm, rightMargin=18 * mm,
                                topMargin=16 * mm, bottomMargin=16 * mm)
        ss = getSampleStyleSheet()
        title = ParagraphStyle("t", parent=ss["Title"], fontSize=20, textColor=colors.HexColor("#3d405b"))
        h = ParagraphStyle("h", parent=ss["Heading2"], fontSize=13, textColor=colors.HexColor("#e07a5f"), spaceBefore=10)
        body = ParagraphStyle("b", parent=ss["BodyText"], fontSize=11, leading=15)
        small = ParagraphStyle("s", parent=ss["BodyText"], fontSize=9, textColor=colors.grey)
        rb = ParagraphStyle("rb", parent=ss["BodyText"], fontSize=9, leading=12, textColor=colors.HexColor("#3d405b"))
        rh = ParagraphStyle("rh", parent=rb, textColor=colors.white, fontName="Helvetica-Bold")

        when = datetime.now(timezone.utc).astimezone().strftime("%d %B %Y, %H:%M")
        mins = max(1, int(round(m["duration_s"] / 60.0)))
        story = []
        story.append(Paragraph("GEETA maths session report", title))
        story.append(Paragraph(f"for {_x(str(m.get('kid') or 'your child'))}  -  {when}", small))
        story.append(Spacer(1, 8))

        # what was learned
        story.append(Paragraph("What was worked on", h))
        year = m.get("year")
        story.append(Paragraph(
            f"Today's focus was <b>{_x(self._skill_line())}</b>"
            + (f" (Year {year})" if year else "")
            + (". " + ("Your child worked it out themselves - well done!" if m["solved"]
                        else "They got a few hints along the way and kept going - that is exactly how learning happens.")
               if m["hints_used"] or m["solved"] else ""), body))

        # per-question breakdown (worksheet sessions)
        items = m.get("items") or []
        if items:
            from geeta.worksheet import MISTAKE_TEXT
            story.append(Paragraph("Questions from the homework page", h))
            qrows = [[Paragraph(c, rh) for c in ("Question", "Skill", "Result", "Help needed")]]
            for it in items:
                lab = it["label"]
                lab = ("Write-the-number " + lab.split()[-1]) if lab.startswith("word") else (
                    "Typed question" if lab == "typed" else f"Question {lab}")
                help_n = int(it.get("hints") or 0) + len(it.get("mistakes") or [])
                qrows.append([Paragraph(_x(lab), rb), Paragraph(_x(it["skill"]), rb),
                              Paragraph("Solved" if it.get("solved") else "Started", rb),
                              Paragraph("none - all by themselves!" if help_n == 0 else
                                        f"{it.get('hints') or 0} hint(s), {len(it.get('mistakes') or [])} slip(s) fixed", rb)])
            qt = Table(qrows, colWidths=[30 * mm, 70 * mm, 22 * mm, 42 * mm])
            qt.setStyle(TableStyle([
                ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#3d405b")),
                ("GRID", (0, 0), (-1, -1), 0.6, colors.HexColor("#dddddd")),
                ("VALIGN", (0, 0), (-1, -1), "TOP"),
            ]))
            story.append(qt)
            cats = {}
            from geeta.worksheet import MISTAKE_QUIET
            for it in items:
                for c in it.get("mistakes") or []:
                    if c not in MISTAKE_QUIET:
                        cats[c] = cats.get(c, 0) + 1
            if cats:
                story.append(Spacer(1, 4))
                story.append(Paragraph(
                    "<b>Worth a little extra practice:</b> "
                    + "; ".join(MISTAKE_TEXT.get(c, c) for c, _ in sorted(cats.items(), key=lambda kv: -kv[1]))
                    + ". These are normal slips - GEETA caught each one and they fixed it themselves.", body))

        # ACARA-style achievement rubric (session 13, user request): parents
        # already read this "working towards / you've done it / above and
        # beyond" grid + A-E letter shape on real school reports - GEETA's
        # own metrics are mapped into it (see derive_grade docstring for the
        # band definitions; not a claim of school-endorsed assessment).
        g = derive_grade(m, self.cfg)
        story.append(Paragraph("Achievement", h))
        band_cols = ["working_towards", "done_it", "above_beyond"]
        headers = ["Success criteria (ACARA)", "Working towards\nE - D", "You've Done It\nC", "Above and Beyond\nB - A"]
        row = [g["acara_code"] or g["skill_name"],
               _band_descriptor(g["desc"], "working_towards"),
               _band_descriptor(g["desc"], "done_it"),
               _band_descriptor(g["desc"], "above_beyond")]
        rubric_rows = [[Paragraph(c, rh) for c in headers],
                       [Paragraph(f"<b>{_x(str(row[0]))}</b>", rb)] + [Paragraph(c, rb) for c in row[1:]]]
        rt = Table(rubric_rows, colWidths=[38 * mm, 42 * mm, 42 * mm, 42 * mm])
        hit_col = 1 + band_cols.index(g["band"])
        rt_style = [
            ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#3d405b")),
            ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
            ("GRID", (0, 0), (-1, -1), 0.6, colors.HexColor("#dddddd")),
            ("VALIGN", (0, 0), (-1, -1), "TOP"),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
            ("TOPPADDING", (0, 0), (-1, -1), 6),
            ("BACKGROUND", (hit_col, 1), (hit_col, 1), colors.HexColor("#fbe8df")),
        ]
        rt.setStyle(TableStyle(rt_style))
        story.append(rt)
        story.append(Spacer(1, 4))
        story.append(Paragraph(
            f"<b>Overall grade: {g['letter']}</b>  &nbsp;&nbsp;  "
            f"<b>Attitude to learning:</b> {g['attitude']}", body))
        story.append(Paragraph(
            "GEETA's own grading, in a familiar shape - not a school assessment. "
            "Based on hints needed, resets, and whether the problem was solved this session.", small))
        story.append(Spacer(1, 6))

        # time + confidence + engagement table
        story.append(Paragraph("How it went", h))
        conf_pct = int(round(m["confidence"] * 100))
        eng_pct = int(round(m["engagement"] * 100))
        rows = [
            ["Time spent", f"about {mins} minute{'s' if mins != 1 else ''}"],
            ["Confidence", f"{conf_pct}%  " + ("(worked it out with little help)" if conf_pct >= 80
                                                else "(needed some hints - normal!)" if conf_pct >= 55
                                                else "(a tricky one - we'll revisit it)")],
            ["Engagement", f"{eng_pct}%  " + ("stayed with it the whole time" if eng_pct >= 70
                                               else "drifted a little, came back" if eng_pct >= 40
                                               else "a short session today")],
            ["Hints used", f"{m['hints_used']} (deepest: {m['max_hint_depth']})"],
        ]
        if m["solved"]:
            rows.append(["Result", "Solved it themselves"])
        focus = m.get("focus")
        if focus:
            pct = focus["engaged_pct"]
            note = ("stayed focused the whole time" if pct >= 90
                    else "focused most of the time, a couple of little wanders" if pct >= 65
                    else "found it hard to stay focused today - that's okay, we gently brought them back")
            extra = f", {focus['refocus_moments']} gentle refocus moment(s)" if focus["refocus_moments"] else ""
            rows.append(["Focus", f"{pct}% {note}{extra}"])
        # Session 13 fix: plain strings in a reportlab Table cell don't wrap -
        # fine while every value was short, but the new Focus row's text is
        # long enough to overflow off the page edge instead of wrapping.
        # Wrap every cell in a Paragraph (same fix already used for the
        # Achievement rubric table above) so long values wrap properly.
        rows = [[Paragraph(f"<b>{r[0]}</b>", rb), Paragraph(r[1], rb)] for r in rows]
        t = Table(rows, colWidths=[38 * mm, 127 * mm])
        t.setStyle(TableStyle([
            ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
            ("TOPPADDING", (0, 0), (-1, -1), 4),
            ("LINEBELOW", (0, 0), (-1, -2), 0.4, colors.HexColor("#dddddd")),
        ]))
        story.append(t)

        # charts
        events = m.get("_events") or []
        for png in (_chart_hint_depth(events), _chart_timeline(events)):
            if png:
                img = Image(io.BytesIO(png))
                img.drawHeight = img.drawHeight * (165 * mm) / img.drawWidth   # keep aspect
                img.drawWidth = 165 * mm
                img.hAlign = "CENTER"
                story.append(Spacer(1, 8))
                story.append(img)

        # progress across sessions (portal history)
        hist = self.history
        if hist and hist.get("sessions"):
            story.append(Paragraph("Progress so far", h))
            story.append(Paragraph(
                f"{hist['sessions']} session(s) with GEETA (including today), "
                + (f"about {hist['minutes']} minute(s) in total, " if hist['minutes'] else "")
                + f"{hist['solved']} question(s) solved.", body))
            prow = [[Paragraph(c, rh) for c in ("Skill", "Solved / tried", "Where they are")]]
            for sk in hist["skills"][:8]:
                prow.append([Paragraph(_x(sk["skill"]), rb), Paragraph(f"{sk['solved']} / {sk['tried']}", rb),
                             Paragraph({"confident": "Confident", "getting there": "Getting there",
                                        "still learning": "Still learning"}.get(sk["status"], sk["status"]), rb)])
            pt = Table(prow, colWidths=[95 * mm, 30 * mm, 40 * mm])
            pt.setStyle(TableStyle([
                ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#3d405b")),
                ("GRID", (0, 0), (-1, -1), 0.6, colors.HexColor("#dddddd")),
            ]))
            story.append(pt)

        story.append(Spacer(1, 10))
        story.append(Paragraph(
            "GEETA never gives the final answer - it guides, and a separate maths engine checks every step. "
            "Only a short learning summary (skills, results, hints) is kept in your GEETA account on the "
            "family/tutor computer - never the chat, photos or camera. You can delete it any time from the portal.", small))
        doc.build(story)
        return buf.getvalue()

    def save(self, path) -> Path:
        p = Path(path)
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_bytes(self.build())
        return p


def render_parent_pdf(session, cfg: dict, path, history: dict | None = None) -> Path:
    """Convenience: build + save the parent PDF for a session."""
    return ParentReport(session, cfg, history=history).save(path)


In [ ]:
# ---------------------------------------------------------------------------
# Dev report (JSON) + delivery hooks (DRY-RUN only)
# ---------------------------------------------------------------------------

class DevReport:
    """Dev-facing metrics (brief C8): tokens/s, TTFT, verifier pass rate,
    hint depth per problem, retries, OCR corrections by kid, leak-guard triggers."""

    def __init__(self, session, cfg: dict):
        self.cfg = cfg
        self.m = derive_metrics(session, cfg)

    def to_dict(self) -> dict:
        m = self.m
        return {
            "ts": datetime.now(timezone.utc).isoformat(),
            "kid": m["kid"],
            "problem_id": m["problem_id"],
            "skill_id": m["skill_id"],
            "solved": m["solved"],
            "duration_s": m["duration_s"],
            "tokens_per_s": m.get("tokens_per_s"),      # filled by serving layer when available
            "ttft_ms": m.get("ttft_ms"),               # filled by serving layer when available
            "verifier_pass_rate": m["verifier_pass_rate"],
            "verifier_checks": m["verifier_checks"],
            "max_hint_depth": m["max_hint_depth"],
            "hints_used": m["hints_used"],
            "retries": m.get("retries", 0),
            "leak_guard_triggers": m["leak_triggers"],
            "ocr_corrections_by_kid": m["ocr_corrections_by_kid"],
            "reengagements": m["reengagements"],
            "turns_per_min": m["turns_per_min"],
        }

    def save_json(self, path) -> Path:
        p = Path(path)
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(json.dumps(self.to_dict(), indent=1) + chr(10))
        return p


def send_email_dryrun(to: str, subject: str, body: str, pdf_path, outdir) -> dict:
    """Build the exact email payload and write it to disk. NEVER sends.

    At pilot time this becomes SES/SMTP: same payload, real transport.
    """
    out = Path(outdir)
    out.mkdir(parents=True, exist_ok=True)
    pdf = Path(pdf_path)
    payload = {
        "dry_run": True,
        "transport": "smtp (SES) - NOT SENT",
        "to": to,
        "subject": subject,
        "body": body,
        "attachments": [pdf.name] if pdf.exists() else [],
        "attachment_sha256": _sha256_file(pdf) if pdf.exists() else None,
    }
    p = out / ("email_%s.json" % datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
    p.write_text(json.dumps(payload, indent=1) + chr(10))
    return {"dry_run": True, "written": str(p)}


def send_whatsapp_dryrun(to: str, text: str, outdir) -> dict:
    """Build the exact WhatsApp Business API payload and write it to disk. NEVER sends."""
    out = Path(outdir)
    out.mkdir(parents=True, exist_ok=True)
    payload = {
        "dry_run": True,
        "transport": "whatsapp-business (Meta/Twilio) - NOT SENT",
        "to": to,
        "type": "text",
        "text": text,
    }
    p = out / ("whatsapp_%s.json" % datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
    p.write_text(json.dumps(payload, indent=1) + chr(10))
    return {"dry_run": True, "written": str(p)}


def _sha256_file(path: Path) -> str:
    import hashlib
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def deliver_parent_report(session, cfg: dict, outdir,
                          parent_email: str | None = None,
                          parent_phone: str | None = None,
                          history: dict | None = None) -> dict:
    """Full C8 delivery flow for one session (dry-run safe).

    Writes: parent PDF + dev JSON + email payload + whatsapp payload.
    Returns a manifest of everything written (all dry-run; nothing sent).
    """
    out = Path(outdir)
    # the child's name is user input: never let it shape a file path
    # ("../../x" or "a/b" used to escape / create folders)
    import re as _re
    kid = _re.sub(r"[^A-Za-z0-9_-]+", "_", str(derive_metrics(session, cfg).get("kid") or "kid")).strip("_")[:32] or "kid"
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

    pdf = render_parent_pdf(session, cfg, out / "parent" / f"{kid}_{stamp}.pdf", history=history)
    dev = DevReport(session, cfg).save_json(out / "dev" / f"{kid}_{stamp}.json")

    m = derive_metrics(session, cfg)
    mins = max(1, int(round(m["duration_s"] / 60.0)))
    body = (f"Hi, here is {m.get('kid') or 'your child'}'s GEETA maths session report. "
            f"They spent about {mins} minute{'s' if mins != 1 else ''} on "
            f"{(m.get('skill_id') or 'maths').replace('_', ' ')}. "
            + ("They solved it themselves - great work!" if m["solved"] else "They practised with a few hints - keep encouraging!"))

    results = {"pdf": str(pdf), "dev_json": str(dev)}
    if parent_email:
        results["email"] = send_email_dryrun(parent_email, "GEETA maths session report", body, pdf, out / "delivery")
    if parent_phone:
        results["whatsapp"] = send_whatsapp_dryrun(parent_phone, body, out / "delivery")
    return results


In [ ]:
# ---------------------------------------------------------------------------
# Self-test: mock session -> parent PDF + dev JSON + dry-run delivery
#
# Gated so importing geeta.reports never runs it. Run explicitly with:
#   GEETA_RUN_SELFTEST=1  (or as a standalone notebook).
# ---------------------------------------------------------------------------
import os as _os
if _os.environ.get("GEETA_RUN_SELFTEST") == "1" or __name__ == "__main__":
    import sys, tempfile
    from geeta.tutor_engine import TutorSession
    from geeta.model_client import MockModel, load_config

    cfg = load_config(_ROOT)
    problem = {
        "id": "selftest_001", "skill_id": "number_addition", "year": 2,
        "strand": "Number", "text": "27 + 45 = ?",
        "answer": "72",
        "hint_ladder": ["Start with the tens: 20 + 40.", "Now the ones: 7 + 5 = 12, that's 1 ten and 2 ones.",
                        "Put the tens together: 60 + 12."],
    }
    model = MockModel(problem)
    sess = TutorSession(cfg, model, kid_name="Ava", problem=problem)
    r = sess.feed_text("I don't know how to start")
    r = sess.feed_text("do the tens first?")
    r = sess.feed_text("20+40 is 60, then 7+5 is 12, so 60+12 = 72!")
    summary = sess.close()

    out = Path(tempfile.mkdtemp(prefix="geeta_reports_"))
    manifest = deliver_parent_report(sess, cfg, out,
                                     parent_email="parent@example.invalid",
                                     parent_phone="+61400000000")
    for k, v in manifest.items():
        print(f"  {k}: {v if isinstance(v, str) else v.get('written')}")
    print("SELFTEST OK - parent PDF + dev JSON + dry-run payloads written to", out)
    print("metrics:", json.dumps(derive_metrics(summary, cfg), indent=1)[:400])
